In [25]:
import pandas as pd
import numpy as np
from ast import literal_eval
from tqdm import tqdm

tqdm.pandas()

In [26]:

"""
Импорт библиотек и загрузка данных.
Загружаем csv-файлы с метаданными, титрами и ключевыми словами.
При чтении movies_metadata.csv игнорируем предупреждения о типах (DtypeWarning).
"""
movies_df = pd.read_csv('data/movies_metadata.csv', low_memory=False)
credits_df = pd.read_csv('data/credits.csv')
keywords_df = pd.read_csv('data/keywords.csv')

In [27]:
"""
Предобработка поля ID.
Удаляем строки с некорректными значениями ID (не числами) в метаданных.
Приводим столбец ID к целочисленному типу во всех таблицах для корректного объединения.
"""
movies_df = movies_df[pd.to_numeric(movies_df['id'], errors='coerce').notnull()]
movies_df['id'] = movies_df['id'].astype(int)
credits_df['id'] = credits_df['id'].astype(int)
keywords_df['id'] = keywords_df['id'].astype(int)

In [28]:
"""
Слияние таблиц.
Объединяем (merge) три датафрейма по общему столбцу 'id'.
"""
df = movies_df.merge(credits_df, on='id').merge(keywords_df, on='id')

In [29]:
"""
Парсинг строковых данных.
Столбцы genres, cast, crew, keywords содержат списки словарей в виде строк.
Преобразуем их в объекты Python с помощью literal_eval.
"""
features = ['cast', 'crew', 'keywords', 'genres']
for feature in features:
    df[feature] = df[feature].progress_apply(literal_eval)

100%|██████████| 46628/46628 [00:00<00:00, 59006.46it/s]


In [30]:
"""
Функция для извлечения имени режиссера из списка crew.
Ищем элемент с job равен 'Director'.
"""
def get_director(x):
    for i in x:
        if i['job'] == 'Director':
            return i['name']
    return np.nan

"""
Извлечение данных из списков.
Для genres и keywords извлекаем полный список названий.
Для cast берем только топ-3 актеров.
Применяем функцию get_director для crew.
"""
df['director'] = df['crew'].progress_apply(get_director)
df['cast'] = df['cast'].progress_apply(lambda x: [i['name'] for i in x][:3] if isinstance(x, list) else [])
df['genres'] = df['genres'].progress_apply(lambda x: [i['name'] for i in x] if isinstance(x, list) else [])
df['keywords'] = df['keywords'].progress_apply(lambda x: [i['name'] for i in x] if isinstance(x, list) else [])

"""
Заполнение пропусков и создание признаков для NLP.
Заполняем пустые значения в overview пустой строкой.
Создаем столбец text_features, объединяя:
- очищенный текст описания (overview)
- жанры
- топ-3 актеров
- имя режиссера
- ключевые слова
Все объединяется через пробел.
"""
df['overview'] = df['overview'].fillna('')

def create_soup(x):
    return ''.join(x['overview']) + ' ' + ' '.join(x['genres']) + ' ' + ' '.join(x['cast']) + ' ' + (str(x['director']) if pd.notna(x['director']) else '') + ' ' + ' '.join(x['keywords'])

df['text_features'] = df.progress_apply(create_soup, axis=1)

100%|██████████| 46628/46628 [00:00<00:00, 61233.48it/s]


In [31]:
"""
Вывод первых строк итоговой таблицы.
Отображаем только запрошенные колонки.
"""
cols = ['id', 'title', 'overview', 'genres', 'cast', 'director', 'keywords', 'text_features']
df[cols].head()

,id,title,overview,genres,cast,director,keywords,text_features
0,862,Toy Story,"Led by Woody, Andy's toys live happily in his ...","[Animation, Comedy, Family]","[Tom Hanks, Tim Allen, Don Rickles]",John Lasseter,"[jealousy, toy, boy, friendship, friends, riva...","Led by Woody, Andy's toys live happily in his ..."
1,8844,Jumanji,When siblings Judy and Peter discover an encha...,"[Adventure, Fantasy, Family]","[Robin Williams, Jonathan Hyde, Kirsten Dunst]",Joe Johnston,"[board game, disappearance, based on children'...",When siblings Judy and Peter discover an encha...
2,15602,Grumpier Old Men,A family wedding reignites the ancient feud be...,"[Romance, Comedy]","[Walter Matthau, Jack Lemmon, Ann-Margret]",Howard Deutch,"[fishing, best friend, duringcreditsstinger, o...",A family wedding reignites the ancient feud be...
3,31357,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...","[Comedy, Drama, Romance]","[Whitney Houston, Angela Bassett, Loretta Devine]",Forest Whitaker,"[based on novel, interracial relationship, sin...","Cheated on, mistreated and stepped on, the wom..."
4,11862,Father of the Bride Part II,Just when George Banks has recovered from his ...,[Comedy],"[Steve Martin, Diane Keaton, Martin Short]",Charles Shyer,"[baby, midlife crisis, confidence, aging, daug...",Just when George Banks has recovered from his ...


In [32]:
"""
Сохранение подготовленного датафрейма в CSV файл.
Файл будет использоваться в следующем ноутбуке для генерации эмбеддингов.
"""
df.to_csv('data/movies_cleaned.csv', index=False)